## Import Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import seaborn as sns
import BanglaProcess as bp

In [2]:
# Warnings supression
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## Loadin Datasets

In [ ]:
train_data_path = 'blp25/dataset/1C/train.tsv'
dev_data_path = 'blp25/dataset/1C/dev.tsv'
dev_test_data_path = 'blp25/dataset/1C/dev_test.tsv'

In [4]:
train_df = pd.read_csv(train_data_path, sep='\t', keep_default_na=False)
dev_df = pd.read_csv(dev_data_path, sep='\t', keep_default_na=False)
dev_test_df = pd.read_csv(dev_test_data_path, sep='\t', keep_default_na=False)

## Frequency Length Distribution

In [5]:
def add_text_length_column(df):
  """
  Adds a 'text_length' column to the DataFrame containing the length of the 'text' column.

  Args:
    df (pandas.DataFrame): The input DataFrame with a 'text' column.

  Returns:
    pandas.DataFrame: The DataFrame with the added 'text_length' column.
  """
  df['text_length'] = df['text'].apply(len)
  return df

train_df = add_text_length_column(train_df)
dev_df = add_text_length_column(dev_df)


## Load and preprocess data for Transformer


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# 1. Define dictionaries to map labels to integer IDs
hate_type_map = {label: i for i, label in enumerate(train_df['hate_type'].unique())}
hate_severity_map = {label: i for i, label in enumerate(train_df['hate_severity'].unique())}
to_whom_map = {label: i for i, label in enumerate(train_df['to_whom'].unique())}

# 2. Replace string labels with integer IDs
train_df['hate_type'] = train_df['hate_type'].map(hate_type_map)
train_df['hate_severity'] = train_df['hate_severity'].map(hate_severity_map)
train_df['to_whom'] = train_df['to_whom'].map(to_whom_map)

dev_df['hate_type'] = dev_df['hate_type'].map(hate_type_map)
dev_df['hate_severity'] = dev_df['hate_severity'].map(hate_severity_map)
dev_df['to_whom'] = dev_df['to_whom'].map(to_whom_map)

# 3. Load the banglabert tokenizer
tokenizer = AutoTokenizer.from_pretrained('google/muril-large-cased')

# 4. Tokenize the 'text' column
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

train_tokenized = train_df.apply(tokenize_function, axis=1)
dev_tokenized = dev_df.apply(tokenize_function, axis=1)
dev_test_tokenized = dev_test_df.apply(tokenize_function, axis=1)

# Convert the tokenized results into a format suitable for PyTorch
train_encodings = list(train_tokenized)
dev_encodings = list(dev_tokenized)
dev_test_encodings = list(dev_test_tokenized)


# 5. Create custom PyTorch Dataset classes
class HateSpeechDataset(Dataset):
    def __init__(self, encodings, hate_type_labels=None, hate_severity_labels=None, to_whom_labels=None):
        self.encodings = encodings
        self.hate_type_labels = hate_type_labels
        self.hate_severity_labels = hate_severity_labels
        self.to_whom_labels = to_whom_labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val) for key, val in self.encodings[idx].items()}
        if self.hate_type_labels is not None:
            item['hate_type_labels'] = torch.tensor(self.hate_type_labels[idx])
        if self.hate_severity_labels is not None:
            item['hate_severity_labels'] = torch.tensor(self.hate_severity_labels[idx])
        if self.to_whom_labels is not None:
            item['to_whom_labels'] = torch.tensor(self.to_whom_labels[idx])
        return item

    def __len__(self):
        return len(self.encodings)

# 6. Instantiate the custom Dataset classes
train_dataset = HateSpeechDataset(train_encodings, train_df['hate_type'].tolist(), train_df['hate_severity'].tolist(), train_df['to_whom'].tolist())
dev_dataset = HateSpeechDataset(dev_encodings, dev_df['hate_type'].tolist(), dev_df['hate_severity'].tolist(), dev_df['to_whom'].tolist())
dev_test_dataset = HateSpeechDataset(dev_test_encodings)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_dataloader = DataLoader(dev_dataset, batch_size=16)
dev_test_dataloader = DataLoader(dev_test_dataset, batch_size=16)

## Define multitask model


In [7]:
import torch.nn as nn
from transformers import AutoModel

class MultitaskModel(nn.Module):
    def __init__(self, muril_model, num_hate_type_labels, num_hate_severity_labels, num_to_whom_labels):
        super().__init__()
        self.muril = muril_model
        hidden_size = muril_model.config.hidden_size

        self.hate_type_classifier = nn.Linear(hidden_size, num_hate_type_labels)
        self.hate_severity_classifier = nn.Linear(hidden_size, num_hate_severity_labels)
        self.to_whom_classifier = nn.Linear(hidden_size, num_to_whom_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, hate_type_labels=None, hate_severity_labels=None, to_whom_labels=None):
        outputs = self.muril(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        # Use the hidden state of the first token (CLS token) instead of pooler_output
        # pooled_output = outputs.last_hidden_state[:, 0]
        pooled_output = outputs.pooler_output

        hate_type_logits = self.hate_type_classifier(pooled_output)
        hate_severity_logits = self.hate_severity_classifier(pooled_output)
        to_whom_logits = self.to_whom_classifier(pooled_output)

        loss = {}
        if hate_type_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss['hate_type_loss'] = loss_fct(hate_type_logits.view(-1, self.hate_type_classifier.out_features), hate_type_labels.view(-1))
        if hate_severity_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss['hate_severity_loss'] = loss_fct(hate_severity_logits.view(-1, self.hate_severity_classifier.out_features), hate_severity_labels.view(-1))
        if to_whom_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss['to_whom_loss'] = loss_fct(to_whom_logits.view(-1, self.to_whom_classifier.out_features), to_whom_labels.view(-1))

        return {
            'hate_type_logits': hate_type_logits,
            'hate_severity_logits': hate_severity_logits,
            'to_whom_logits': to_whom_logits,
            'loss': loss
        }

# Load the pre-trained MuRIL model (matching the tokenizer)
muril_model = AutoModel.from_pretrained('ai4bharat/IndicBERTv2-MLM-only')

# Get the number of unique labels from the mappings
num_hate_type_labels = len(hate_type_map)
num_hate_severity_labels = len(hate_severity_map)
num_to_whom_labels = len(to_whom_map)

# Instantiate the custom multitask model
model = MultitaskModel(muril_model, num_hate_type_labels, num_hate_severity_labels, num_to_whom_labels)

print("Multitask model instantiated with custom classification heads.")
print("Number of hate_type labels:", num_hate_type_labels)
print("Number of hate_severity labels:", num_hate_severity_labels)
print("Number of to_whom labels:", num_to_whom_labels)

Multitask model instantiated with custom classification heads.
Number of hate_type labels: 6
Number of hate_severity labels: 3
Number of to_whom labels: 5


## Set up training


In [8]:
from torch.optim import AdamW

# Validate labels before training to catch potential issues
print("Validating labels...")
print(f"Train labels - hate_type: min={train_df['hate_type'].min()}, max={train_df['hate_type'].max()}")
print(f"Train labels - hate_severity: min={train_df['hate_severity'].min()}, max={train_df['hate_severity'].max()}")
print(f"Train labels - to_whom: min={train_df['to_whom'].min()}, max={train_df['to_whom'].max()}")

print(f"Dev labels - hate_type: min={dev_df['hate_type'].min()}, max={dev_df['hate_type'].max()}")
print(f"Dev labels - hate_severity: min={dev_df['hate_severity'].min()}, max={dev_df['hate_severity'].max()}")
print(f"Dev labels - to_whom: min={dev_df['to_whom'].min()}, max={dev_df['to_whom'].max()}")

print(f"Expected ranges - hate_type: 0-{num_hate_type_labels-1}, hate_severity: 0-{num_hate_severity_labels-1}, to_whom: 0-{num_to_whom_labels-1}")

# Check for any NaN or invalid values
print(f"NaN values in train - hate_type: {train_df['hate_type'].isna().sum()}, hate_severity: {train_df['hate_severity'].isna().sum()}, to_whom: {train_df['to_whom'].isna().sum()}")
print(f"NaN values in dev - hate_type: {dev_df['hate_type'].isna().sum()}, hate_severity: {dev_df['hate_severity'].isna().sum()}, to_whom: {dev_df['to_whom'].isna().sum()}")

optimizer = AdamW(model.parameters(), lr=1e-5)

print("AdamW optimizer instantiated.")

Validating labels...
Train labels - hate_type: min=0, max=5
Train labels - hate_severity: min=0, max=2
Train labels - to_whom: min=0, max=4
Dev labels - hate_type: min=0, max=5
Dev labels - hate_severity: min=0, max=2
Dev labels - to_whom: min=0, max=4
Expected ranges - hate_type: 0-5, hate_severity: 0-2, to_whom: 0-4
NaN values in train - hate_type: 0, hate_severity: 0, to_whom: 0
NaN values in dev - hate_type: 0, hate_severity: 0, to_whom: 0
AdamW optimizer instantiated.


## Train the model


In [9]:
from tqdm.auto import tqdm
import gc

# 1. Define the number of training epochs.
num_epochs = 3  # You can adjust this number

# 2. Set the device to GPU if available, otherwise use CPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 3. Move the model to the selected device.
model.to(device)

# Initialize best validation loss for saving the model
best_val_loss = float('inf')

# 4. Iterate through the epochs.
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # 5. For each epoch, set the model to training mode and iterate through the train_dataloader.
    model.train()
    train_loss = 0.0
    
    try:
        for batch_idx, batch in enumerate(tqdm(train_dataloader, desc="Training")):
            # Move batch data to the device
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Debug: Check batch content occasionally
            if batch_idx == 0:
                print(f"First batch shapes: input_ids: {batch['input_ids'].shape}")
                print(f"Label ranges in batch - hate_type: {batch['hate_type_labels'].min()}-{batch['hate_type_labels'].max()}")
                print(f"Label ranges in batch - hate_severity: {batch['hate_severity_labels'].min()}-{batch['hate_severity_labels'].max()}")
                print(f"Label ranges in batch - to_whom: {batch['to_whom_labels'].min()}-{batch['to_whom_labels'].max()}")

            # Perform forward pass
            outputs = model(**batch)

            # Calculate the total loss (sum of individual task losses)
            total_loss = sum(outputs['loss'].values())

            # Perform backpropagation
            optimizer.zero_grad()
            total_loss.backward()

            # Update the model parameters
            optimizer.step()

            train_loss += total_loss.item()

    except Exception as e:
        print(f"Error during training at batch {batch_idx}: {e}")
        print("Setting CUDA_LAUNCH_BLOCKING=1 for better debugging...")
        import os
        os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
        raise

    avg_train_loss = train_loss / len(train_dataloader)

    # 7. After each training epoch, set the model to evaluation mode and iterate through the dev_dataloader.
    model.eval()
    val_hate_type_loss = 0.0
    val_hate_severity_loss = 0.0
    val_to_whom_loss = 0.0
    val_total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(dev_dataloader, desc="Validation"):
            # Move batch data to the device
            batch = {k: v.to(device) for k, v in batch.items()}

            # Perform forward pass to get the logits and losses
            outputs = model(**batch)

            # Accumulate the validation losses for each task
            val_hate_type_loss += outputs['loss']['hate_type_loss'].item()
            val_hate_severity_loss += outputs['loss']['hate_severity_loss'].item()
            val_to_whom_loss += outputs['loss']['to_whom_loss'].item()
            val_total_loss += sum(outputs['loss'].values()).item()


    # 9. Calculate the average validation loss for each task and the total average validation loss for the epoch.
    avg_val_hate_type_loss = val_hate_type_loss / len(dev_dataloader)
    avg_val_hate_severity_loss = val_hate_severity_loss / len(dev_dataloader)
    avg_val_to_whom_loss = val_to_whom_loss / len(dev_dataloader)
    avg_val_total_loss = val_total_loss / len(dev_dataloader)

    # 10. Print the training and validation losses for each epoch.
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Losses:")
    print(f"  Hate Type: {avg_val_hate_type_loss:.4f}")
    print(f"  Hate Severity: {avg_val_hate_severity_loss:.4f}")
    print(f"  To Whom: {avg_val_to_whom_loss:.4f}")
    print(f"  Total Validation Loss: {avg_val_total_loss:.4f}")

    # 11. (Optional) Save the model checkpoint if the total validation loss improves.
    if avg_val_total_loss < best_val_loss:
        best_val_loss = avg_val_total_loss
        torch.save(model.state_dict(), 'best_multitask_model.pth')
        print("Validation loss improved. Model checkpoint saved.")

    # Clear cache
    if 'batch' in locals():
        del batch
    if 'outputs' in locals():
        del outputs
    if 'total_loss' in locals():
        del total_loss
    if torch.cuda.is_available():
      torch.cuda.empty_cache()
    gc.collect()

print("\nTraining finished.")

Using device: cuda

Epoch 1/3


Training:   0%|          | 0/2221 [00:00<?, ?it/s]

First batch shapes: input_ids: torch.Size([16, 128])
Label ranges in batch - hate_type: 0-2
Label ranges in batch - hate_severity: 0-2
Label ranges in batch - to_whom: 0-4


Validation:   0%|          | 0/157 [00:00<?, ?it/s]

Train Loss: 3.2245
Validation Losses:
  Hate Type: 1.1251
  Hate Severity: 0.8278
  To Whom: 1.1471
  Total Validation Loss: 3.0999
Validation loss improved. Model checkpoint saved.

Epoch 2/3


...

Train Loss: 3.2539
Validation Losses:
  Hate Type: 1.1657
  Hate Severity: 0.8477
  To Whom: 1.1834
  Total Validation Loss: 3.1968

Epoch 3/3


Training:   0%|          | 0/2221 [00:00<?, ?it/s]

First batch shapes: input_ids: torch.Size([16, 128])
Label ranges in batch - hate_type: 0-4
Label ranges in batch - hate_severity: 0-2
Label ranges in batch - to_whom: 0-4


Validation:   0%|          | 0/157 [00:00<?, ?it/s]

Train Loss: 3.2148
Validation Losses:
  Hate Type: 1.1347
  Hate Severity: 0.8238
  To Whom: 1.1547
  Total Validation Loss: 3.1131

Training finished.


## Make Predictions

In [10]:
# Inverse mappings to convert integer IDs back to labels
hate_type_inverse_map = {v: k for k, v in hate_type_map.items()}
hate_severity_inverse_map = {v: k for k, v in hate_severity_map.items()}
to_whom_inverse_map = {v: k for k, v in to_whom_map.items()}

# Set the model to evaluation mode
model.eval()

# Lists to store predictions
all_hate_type_preds = []
all_hate_severity_preds = []
all_to_whom_preds = []

# Iterate through the dev_test_dataloader
with torch.no_grad():
    for batch in tqdm(dev_test_dataloader, desc="Predicting"):
        # Move batch data to the device
        batch = {k: v.to(device) for k, v in batch.items()}

        # Perform forward pass to get the logits
        outputs = model(**batch)

        # Get predictions for each task
        hate_type_preds = torch.argmax(outputs['hate_type_logits'], dim=-1)
        hate_severity_preds = torch.argmax(outputs['hate_severity_logits'], dim=-1)
        to_whom_preds = torch.argmax(outputs['to_whom_logits'], dim=-1)

        # Extend the lists with predictions
        all_hate_type_preds.extend(hate_type_preds.cpu().numpy())
        all_hate_severity_preds.extend(hate_severity_preds.cpu().numpy())
        all_to_whom_preds.extend(to_whom_preds.cpu().numpy())

# Convert predicted integer IDs back to original labels
predicted_hate_type_labels = [hate_type_inverse_map[pred] for pred in all_hate_type_preds]
predicted_hate_severity_labels = [hate_severity_inverse_map[pred] for pred in all_hate_severity_preds]
predicted_to_whom_labels = [to_whom_inverse_map[pred] for pred in all_to_whom_preds]

# Create a DataFrame with the predictions and original text
predictions_df = pd.DataFrame({
    'id': dev_test_df['id'], # Include the original IDs
    'text': dev_test_df['text'], # Include original text for context
    'predicted_hate_type': predicted_hate_type_labels,
    'predicted_hate_severity': predicted_hate_severity_labels,
    'predicted_to_whom': predicted_to_whom_labels
})

print("\nPredictions on dev_test data:")
display(predictions_df.head())

Predicting:   0%|          | 0/157 [00:00<?, ?it/s]


Predictions on dev_test data:


,id,text,predicted_hate_type,predicted_hate_severity,predicted_to_whom
0,879187,শুভ কামনা রইল বাংলাদেশ জন্য ইনশাআল্লাহ জয় হবে,None,Little to None,None
1,316919,গোয়া মারা দিয়ে আছে বাংলাদেশ মাদারচোদ নিউজ করে ...,None,Little to None,None
2,916242,ভাইয়া আপনি অভিনেতা হইয়েন না না হলে সবাই বাচ্...,None,Little to None,None
3,786824,আমাদেরো তাই দেখছি,None,Little to None,None
4,47284,পুলিশ কতটা টাকা নিয়ে,None,Little to None,None


## Generate Submission DataFrame

In [11]:
# Create the submission DataFrame by selecting and renaming columns
submission_df = predictions_df[['id', 'predicted_hate_type', 'predicted_hate_severity', 'predicted_to_whom']].copy()
submission_df.rename(columns={
    'predicted_hate_type': 'hate_type',
    'predicted_hate_severity': 'hate_severity',
    'predicted_to_whom': 'to_whom'
}, inplace=True)

# Add the 'model' column with the specified model name
submission_df['model'] = 'multitask-muril-2-epoch'

# Display the first few rows of the submission DataFrame
print("Submission DataFrame:")
display(submission_df.head())

Submission DataFrame:


,id,hate_type,hate_severity,to_whom,model
0,879187,None,Little to None,None,multitask-muril-2-epoch
1,316919,None,Little to None,None,multitask-muril-2-epoch
2,916242,None,Little to None,None,multitask-muril-2-epoch
3,786824,None,Little to None,None,multitask-muril-2-epoch
4,47284,None,Little to None,None,multitask-muril-2-epoch


In [12]:
submission_df.hate_type.value_counts()

hate_type
None    2512
Name: count, dtype: int64

In [13]:
submission_df.hate_severity.value_counts()

hate_severity
Little to None    2512
Name: count, dtype: int64

In [14]:
submission_df.to_whom.value_counts()

to_whom
None    2512
Name: count, dtype: int64

In [15]:
submission_df.shape

(2512, 5)

In [16]:
# Save the submission DataFrame to a TSV file
submission_df.to_csv('subtask_1C.tsv', sep='\t', index=False)

print("Submission file 'multitask_csebuetnlp_banglabert_2_epoch.tsv' created successfully.")

Submission file 'multitask_csebuetnlp_banglabert_2_epoch.tsv' created successfully.
